In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

: 

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

In [ ]:
target = "Retard moyen de tous les trains à l'arrivée"
y = df[target]

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["month"] = df["Date"].dt.month
features = [
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "Nombre de trains annulés",
    "month",
    "Nombre trains en retard > 15min",
    "Nombre trains en retard > 30min",
    "Nombre trains en retard > 60min",
    "Service"
]
X = df[features]

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
baseline_prediction = np.full_like(y_test, y_train.mean())

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline Performance:")
print("RMSE:", baseline_rmse)
print("MAE:", baseline_mae)
print("R²:", baseline_r2)

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\nRandom Forest Performance:")
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

print("\nImprovement over Baseline (RMSE reduction):")
print(baseline_rmse - rmse)

In [ ]:
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Most Important Features:")
print(importances.head(10))

joblib.dump(model, "delay_prediction_model.pkl")
joblib.dump(X.columns, "model_features.pkl")